In [ ]:
!pip install ktrain pymorphy2 conllu -q

# Как работает поиск? (А также немного про классы и красоту кода)

Давайте попробуем разобраться на примере поиска ответа на следующий вопрос:

In [ ]:
question = "Почему у кошки четыре лапки?"

У нас есть список документов, среди которых скорее всего есть подходящие с ответом.

In [ ]:
documents = [
    'У кошек четыре лапки, чтобы можно было быстрее бегать за бабочками',
    'Если бы у собаки было шесть лапок, она бы стала тараканом',
    'Говорят, что кошка имеет четыре лапки, так как это приносит удачу, как четырехлистный клевер',
    'Если у стула четыре ножки, можно ли считать его кошкой?',
    'Сдача проектов по Автобрее 30 октября до 14.00'
]

__Вопрос:__ В каких документах хранится ответ на наш вопрос? Как вы это поняли? Можно ли как-то понять, какой из документов больше подходит под наш запрос?

## Вариант 1: Прямой индекс
Давайте попробуем искать полное вхождение слова в документ.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
from collections import Counter, defaultdict
from nltk import word_tokenize
from typing import List

In [ ]:
class DocumentsDatabase():
    """
    Этот класс создает базу данных на основе прямой индексации
    и осуществляет поиск по ней.
    """
    def __init__(self, documents: List[str]):
        """
        Высчитывает прямой индекс
        :param documents: список документов, которые входят в базу данных
        """
        self.documents = {doc: [word.lower() for word in word_tokenize(doc)] for doc in documents}

    def search(self, question: str) -> List[tuple]:
        """
        Эта функция осуществляет поиск по базе данных на основе запроса
        :param question: строка запроса
        """
        result = Counter()
        for doc, wordlist in self.documents.items():
            for word in word_tokenize(question):
                if word.lower() in wordlist:
                    result[doc] += 1

        return result.most_common()

In [ ]:
help(DocumentsDatabase)

In [ ]:
db = DocumentsDatabase(documents)
db.search(question)

__Вопросы:__
1. Можно ли искать вхождение слова в сыром документе? (не разделённом на слова)
2. Как можно улучшить поиск?

## Вариант 2: Прямой индекс улучшенный

Возвращаемся к старой доброй лемматизации и чистке слов.

In [ ]:
from nltk.corpus import stopwords
from pymorphy2 import MorphAnalyzer

stops = stopwords.words('russian')
morph = MorphAnalyzer()

In [ ]:
class DocumentsDatabase():

    def __init__(self, documents):
        self.documents = self.__create_base(documents)

    def __create_base(self, documents):
        result = {}
        for doc in documents:
            result[doc] = self.prepare_doc(doc)
        return result

    @staticmethod
    def prepare_doc(doc):
        doc = [word.lower() for word in word_tokenize(doc)]
        clean_doc = []
        for word in doc:
            if word.isalpha():
                word = morph.parse(word)[0].normal_form
                if word not in stops:
                    clean_doc.append(word)
        return clean_doc

    def search(self, question):
        question = self.prepare_doc(question)
        result = Counter()
        for doc, wordlist in self.documents.items():
            for word in question:
                if word in wordlist:
                    result[doc] += 1

        return result.most_common()

db = DocumentsDatabase(documents)
db.search(question)

__Вопросы:__
1. Насколько это эффективно? Можно ли быстрее?

## Вариант 3: Обратный индекс

Проходиться по списку слов в предложении очень долго. А что если сделать наоборот?

In [ ]:
class DocumentsDatabaseInverse():

    def __init__(self, documents):
        self.documents = self.__create_base(documents)

    def __create_base(self, documents):
        result = defaultdict(set)
        ### YOUR CODE HERE ###

        return result

    @staticmethod
    def prepare_doc(doc):
        doc = [word.lower() for word in word_tokenize(doc)]
        clean_doc = []
        for word in doc:
            if word.isalpha():
                word = morph.parse(word)[0].normal_form
                if word not in stops:
                    clean_doc.append(word)
        return clean_doc

    def search(self, question):
        question = self.prepare_doc(question)
        result = Counter()
        ### YOUR CODE HERE ###

        return result.most_common()

db_inverse = DocumentsDatabaseInverse(documents)
db_inverse.search(question)

То же самое, только с _наследованием_.

In [ ]:
class DocumentsDatabaseInverse(DocumentsDatabase):

    def __init__(self, documents):
        super().__init__(documents)
        self.documents = self.__create_base(documents)

    def __create_base(self, documents):
        result = defaultdict(set)
        for doc in documents:
            clean_doc = self.prepare_doc(doc)
            for word in clean_doc:
                result[word].add(doc)
        return result

    def search(self, question):
        question = self.prepare_doc(question)
        result = Counter()
        for word in question:
            doclist = self.documents[word]
            for doc in doclist:
                result[doc] += 1

        return result.most_common()

db_inverse = DocumentsDatabaseInverse(documents)
db_inverse.search(question)

__Вопрос:__ а если мы говорим про машинное обучение, то в каком формате удобнее всего хранить данные?

## Вариант 4: векторизация

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances

In [ ]:
class DocumentsDatabaseMatrix():

    def __init__(self, documents):
        self.documents = np.array(documents)
        self.doc_matrix = self.__create_base(documents)

    def __create_base(self, documents):
        self.vectorizer = CountVectorizer()
        result = self.vectorizer.fit_transform(documents)
        return result

    def search(self, question):
        question = self.vectorizer.transform([question])
        dst = cosine_distances(question, self.doc_matrix)

        return self.documents[dst.argsort()[0]]

db_matrix = DocumentsDatabaseMatrix(documents)
db_matrix.search(question)

### Сравнение

In [ ]:
import time

In [ ]:
start = time.time()
for _ in range(10000):
    ans = db.search(question)
end = time.time()
print('Время прямого индекса:', end-start)

In [ ]:
start = time.time()
for _ in range(10000):
    ans = db_inverse.search(question)
end = time.time()
print('Время обратного индекса:', end-start)

In [ ]:
start = time.time()
for _ in range(10000):
    ans = db_matrix.search(question)
end = time.time()
print('Время матричного индекса:', end-start)

# А как лучше всего это хранить?

__Вопрос:__ А какие форматы хранения вы уже знаете?

В целом, я бы сказала, что тут даже важен не _формат_, а то какую именно информацию о данных мы храним. Что именно нам нужно хранить для решения задачи поиска подходящих документов?

### Текстовый файлик, csv/tsv

In [ ]:
import pandas as pd

In [ ]:
documents = [
    'У кошек четыре лапки, чтобы можно было быстрее бегать за бабочками',
    'Если бы у собаки было шесть лапок, она бы стала тараканом',
    'Говорят, что кошка имеет четыре лапки, так как это приносит удачу, как четырехлистный клевер',
    'Если у стула четыре ножки, можно ли считать его кошкой?',
    'Сдача проектов по Автобрее 30 октября до 14.00'
]

authors = [
    'Сайт Зоологии',
    'анектдоты точка ру',
    'Приметы и суеверия',
    'Вопросы меил ру',
    'Ксюша'
]

dates = [
    '01.01.1970',
    '10.09.2100',
    '21.03.2015',
    '14.07.2002',
    '22.10.2024'
]

In [ ]:
def prepare_doc(doc):
    doc = [word.lower() for word in word_tokenize(doc)]
    clean_doc = []
    for word in doc:
        if word.isalpha():
            word = morph.parse(word)[0].normal_form
            if word not in stops:
                clean_doc.append(word)
    return clean_doc

data = pd.DataFrame(data={'text': documents, 'author': authors, 'date': dates}, columns=['text', 'author', 'date'])
data['words'] = data.text.apply(prepare_doc)

In [ ]:
data.words.loc[0]

__Вопрос:__ хорошо ли хранить списки в таблице?

In [ ]:
datapath = 'db_example.tsv'
data.to_csv(datapath, sep='\t')

data_csv = pd.read_csv(datapath, sep='\t')
data_csv.words.loc[0]

In [ ]:
datapath = 'db_example.pickle'
data.to_pickle(datapath)

data_pickle = pd.read_pickle(datapath)
data_pickle.words.loc[0]

Если мы хотим сохранить список слов, то лучше всего тогда сделать его строкой с каким-то разделителем, например "," или ";"

__Вопрос:__ А как сделать таблицу для обратного индекса?

In [ ]:
def create_base(self, documents):
    result = defaultdict(set)
    for doc in documents:
        clean_doc = self.prepare_doc(doc)
        for word in clean_doc:
            result[word].add(doc)
    return result

def prepare_doc(doc):
    doc = [word.lower() for word in word_tokenize(doc)]
    clean_doc = []
    for word in doc:
        if word.isalpha():
            word = morph.parse(word)[0].normal_form
            if word not in stops:
                clean_doc.append(word)
    return clean_doc

data = pd.DataFrame(data={'text': documents, 'author': authors, 'date': dates}, columns=['text', 'author', 'date'])
df_tokens = pd.DataFrame(columns=['token', 'sent_id'])
for i, row in data.iterrows():
    tokens = prepare_doc(row.text)
    for token in tokens:
        df_tokens.loc[len(df_tokens.index)] = [token, i]

In [ ]:
df_tokens.head()

In [ ]:
sent_ids = df_tokens.loc[df_tokens.token=='кошка', 'sent_id'].tolist()
data.loc[sent_ids]

__Вопросы:__
1. Можно ли сделать одну таблицу обратного индекса с предложениями и токенами?
2. Как можно устроить поиск по запросу из нескольких слов по такой таблице?

### Conllu

Можно хранить так же в текстовом формате с табуляцией-разделителем, но уже по правила conllu. Есть два варианта: либо просто записывать в какой-то текстовый файл информацию формата conllu и считать библиотекой то, что получилось, либо воспользоваться инструментами библиотеки.

In [ ]:
from conllu import parse
from conllu.models import TokenList, Token

#### Самостоятельный текст

In [ ]:
data = """
# id = 0
# authors = Ксюша
# text = Сдача проектов по Автобрее 30 октября до 14.00
1   Сдача     сдача    NOUN
2   проектов   проект  NOUN
3   по   по  PREP
4   Автобрее     автобрея    NOUN
5   30   30   NUM
6   октября    октябрь   NOUN
7   до     до    PREP
8   14    14   NUM
9   .     .    PUNCT
10  00       00      NUM

# id = 1
# authors = Николай Дроздов
# text = Если у собаки было шесть лапок, она бы стала тараканом
1   Если   если   CONJ
2   у   у   PREP
3   собака  собака  NOUN
4   было   быть   VERB
5   шесть   шесть   NUM
6   лапок   лапка   NOUN
7   ,   ,   PUNCT
8   она она PRON
9   бы  бы  PART
10  стала   стать   VERB
11  тараканом   такаран NOUN
"""

sentences = parse(data)

In [ ]:
print(type(sentences))
sentences

In [ ]:
sent = sentences[0]
metadata = sent.metadata

In [ ]:
[token for token in sent]

In [ ]:
metadata

In [ ]:
sent.filter(upos='NOUN')

In [ ]:
sent.filter(upos='NOUN', lemma='октябрь')

#### Инструменты библиотеки

In [ ]:
def prepare_doc(doc):
    doc = [word.lower() for word in word_tokenize(doc)]
    clean_doc = []
    for word in doc:
        if word.isalpha():
            word_norm = morph.parse(word)[0].normal_form
            clean_doc.append((word, word_norm))
    return clean_doc

tokens = prepare_doc(documents[1])
sent = []
for i, (token, token_norm) in enumerate(tokens):
    sent.append({
        'id': i,
        'form': token,
        'lemma': token_norm
    })

sent = TokenList(sent)
sent.metadata = {'author': 'Me'}
sent

__Вопрос:__ А как здесь устроить поиск по нескольким словам из запроса?

### Векторизация (если очень хочется)


In [ ]:
from scipy.sparse import save_npz, load_npz

In [ ]:
matrix_path = 'matrix.npz'
matrix_to_save = db_matrix.doc_matrix
save_npz(matrix_path, matrix_to_save)
matrix = load_npz(matrix_path)

In [ ]:
matrix

In [ ]:
features = db_matrix.vectorizer.get_feature_names_out()
idx2word = {i: word for i, word in enumerate(features)}
word2idx = {word: i for i, word in idx2word.items()}

In [ ]:
matrix_np = matrix.toarray()
matrix_np

In [ ]:
cat = word2idx['кошка']
idx = np.where((matrix_np[:, cat] > 0))[0]

In [ ]:
db_matrix.documents[idx[0]]

### Что ещё можно сделать?
- можно сделать таблицу sql (так, например, делала я, когда делала этот проект)
- можно сделать json, аналогично conllu, по сути
- можно сделать несколько таблиц - отдельно предложения и метаданные, отдельно токены и их характеристики (можно текстом, а можно ещё сильнее показать свои навыки и сделать отдельную таблицу с ними и писать айдишники)
- а можно сделать одну таблицу

### Что важно для проекта?
- Самое важное, чтобы работало:)
- А ещё чтобы ошибок не было (максимальное кол-во различных случаев обрабатывалось или чтобы не всё не падало, т.е. обработка ошибок с try-except)
- Чтобы скорость была оптимальная.
- И чтобы юзер-френдли (комментарии, если код не самый простой и понятный, описание классов/функций через докстринги)

Кстати, какую инфу надо хранить о токене, помимо его характеристик, чтобы обрабатывать запрос для проекта?